# Упражнение: Стоимость координации: supervisor, все-со-всеми и конвейер

## Задание

Команда из 7 LLM-агентов решает задачу. Файл `input.csv`: `agent_id`, `name`, `role`
(`orchestrator` — ровно один, остальные `worker`), `msg_tokens` (длина сообщения, которое
агент отправляет за один раз) и `price_per_1k` (цена в долларах за 1000 входных токенов
модели этого агента: платит тот, кто **читает** сообщение).

Сравните три схемы координации. Один раунд устроен так:

- `supervisor` — оркестратор шлёт задачу каждому исполнителю, и каждый исполнитель отвечает оркестратору;
- `all_to_all` — каждый агент отправляет своё сообщение каждому другому агенту;
- `pipeline` — агенты по возрастанию `agent_id` образуют цепочку: каждый отправляет сообщение следующему
  (последний — никому).

Сообщение длиной `msg_tokens` отправителя считается за `msg_tokens` токенов; его стоимость —
`msg_tokens · price_per_1k(получателя) / 1000`. Всего **3 раунда** (схема повторяется без изменений).
Для каждой схемы посчитайте:

- `messages` — число сообщений за 3 раунда;
- `tokens` — суммарное число токенов во всех сообщениях;
- `cost_usd` — суммарная стоимость, округление до 4 знаков;
- `busiest_agent` — `agent_id` агента, получившего больше всех токенов (при равенстве — меньший `agent_id`);
- `busiest_tokens` — сколько токенов он получил за 3 раунда.

## Подсказки

- Представьте каждую схему списком рёбер `(отправитель, получатель)` за один раунд: у `supervisor` их `2·k`,
  у `all_to_all` — `n·(n−1)`, у `pipeline` — `n−1`. Число сообщений за 3 раунда — длина списка · 3.
- Один цикл по рёбрам даёт всё сразу: `tokens += msg[s]·3`, `cost += msg[s]·3·price[r]/1000`,
  `inbox[r] += msg[s]·3`. Оркестратора найдите по `role`, а не по номеру строки.
- Цену берите у получателя, а не у отправителя. Для `busiest_agent` используйте
  `min(ids, key=lambda a: (-inbox[a], a))`.

## Ожидаемый формат выходного файла

CSV с колонками `scheme, messages, tokens, cost_usd, busiest_agent, busiest_tokens` — 3 строки
в порядке `supervisor`, `all_to_all`, `pipeline`.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/multi_agent_systems
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.